## PyINE code execution deltas generation demo

This notebook shows how to trace an example code snippet using the PyINE framework utility functions, and shows how to turn the trace results into deltas.

Note: we are not involving any dataset here, just running on a single manually defined snippet.

In [ ]:
import pyine.data.deltas.dataset_utils
import pyine.utils.code.execution
import pyine.utils.portability
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()  # loads dotenv variables, seeds, sets up logging, etc.

In [ ]:
# define the code snippet we will be using below
example_snippet = """\
def calculate_area(length: float, width: float) -> float:
    '''Returns the area of the rectangle specified via length and width.

    Specifically: returns area = length * width.
    '''
    area = length * width
    return area

length = float(input("Enter the length: "))
width = float(input("Enter the width: "))
area = calculate_area(length, width)
print(f"The area of the rectangle is: {area:.2f} square units")
"""
# input args are required for tracing
example_input_args = """\
5.0
3.0
"""

In [ ]:
# conduct the actual tracing using proper scaffolding to catch everything we need
trace_result = pyine.utils.code.execution.execute_and_trace_code(
    example_snippet,
    example_input_args,
    identifier="dummy",  # required for delta generation, for proper logging
    trace_only_inside_code_string=True,
)

# print the captured trace data below
print("\nTraced code string:")
pyine.utils.portability.print_code_with_numbered_lines(example_snippet, 1)
print("\nTraced steps:")
for traced_step_idx, traced_step in enumerate(trace_result.traced_steps):
    if traced_step is None:
        print(f"\tstep#{traced_step_idx:04d}:\t(out-of-context execution)")
    else:
        print(f"\tstep#{traced_step_idx:04d}:\t{traced_step}")
if trace_result.return_value is not None:
    print(f"\nCaptured return value:\n\t{trace_result.return_value}")
if trace_result.exception is not None:
    print(f"\nCaptured exception:\n\t{trace_result.exception}")
if trace_result.stdout:
    print(f"\nCaptured output:\n\t{trace_result.stdout}")
if trace_result.stderr:
    print(f"\nCaptured error:\n\t{trace_result.stderr}")

In [ ]:
# given the previously generate trace data, compute 'deltas', i.e. stepwise execution variable changes
trace_results_with_deltas = pyine.data.deltas.dataset_utils.get_deltas_from_trace_steps(
    trace_res=trace_result,
    delta_generator=pyine.data.deltas.dataset_utils.DeltaGeneratorType.SIMPLE,
)

# print the resulting deltas below
print(f"traced steps: {len(trace_results_with_deltas.traced_steps)}")
print("deltas:")
for delta_idx, delta in enumerate(trace_results_with_deltas.deltas):
    print(f"\td#{delta_idx}:\t{delta}")